In [ ]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import evaluate, summary, time_inference, time_inference_torch
from gps_ids import protocol
from gps_ids.results import append_validated

import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time
import os

In [ ]:
# Constants
RANDOM_STATE = 1082
NUM_CLASSES = 4
INPUT_SIZE = 1
BATCH_SIZE = 64
HIDDEN_SIZE = 64
NUM_EPOCHS = 100
LEARNING_RATE = 0.001

SYNTH_GENERATORS = ["ctgan", "gc", "tvae"]
SAMPLE_SIZES = [10_000, 100_000, 1_000_000]
SYNTH_FILES = [f"spoofing_{g}_{n}.csv" for g in SYNTH_GENERATORS for n in SAMPLE_SIZES]


In [5]:
# Define LSTM Classifier
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True, num_layers=2)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [6]:
results = []

In [7]:
# Define a basic LSTM model for classification
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True, num_layers=2)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

In [ ]:
for synth_name in SYNTH_FILES:
    try:
        EXP = protocol.synthetic_vs_full_real("2d", synth_name)
    except FileNotFoundError:
        print(f"File {synth_name} not found, skipping.")
        continue

    X_train_scaled = EXP.X_train.values.astype('float32')
    X_real_scaled = EXP.X_test.values.astype('float32')
    y_train = EXP.y_train
    y_real = EXP.y_test

    # Reshape for LSTM
    SEQUENCE_LENGTH = X_train_scaled.shape[1]
    X_train_lstm = X_train_scaled.reshape(-1, SEQUENCE_LENGTH, INPUT_SIZE)
    X_real_lstm = X_real_scaled.reshape(-1, SEQUENCE_LENGTH, INPUT_SIZE)

    # Prepare tensors
    X_train_tensor = torch.tensor(X_train_lstm, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    X_test_tensor = torch.tensor(X_real_lstm, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_real.values, dtype=torch.long)

    # DataLoaders
    train_data = TensorDataset(X_train_tensor, y_train_tensor)
    test_data = TensorDataset(X_test_tensor, y_test_tensor)
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

    # Init model, loss, optimizer
    model = LSTMClassifier(INPUT_SIZE, HIDDEN_SIZE, NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # Training
    start_time = time.time()
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * X_batch.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)
        if (epoch + 1) % 10 == 0 or epoch == 0 or epoch == NUM_EPOCHS - 1:
            epoch_loss = running_loss / len(train_loader.dataset)
            epoch_acc = correct / total
            print(f"{synth_name} | Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}")

    train_time = time.time() - start_time

    # Evaluation
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    print(f"{synth_name} | Accuracy: {acc:.4f} | F1: {f1:.4f}")

    # Store results
    _row = evaluate(
        all_labels,
        all_preds,
        model_name="LSTM Classifier",
        family="Synthetic-LSTM",
        y_proba=all_probs,
        train_time_sec=train_time,
        num_epochs=NUM_EPOCHS,
        parameters=str(model.state_dict().keys()),
        timing=time_inference_torch(model, next(iter(test_loader))[0].to(device)),
        **EXP.eval_kwargs(),
    )
    print(summary(_row))
    results.append(_row)


In [ ]:
from gps_ids.schema import results_path

append_validated(results, str(results_path("2d")))
print(f"Results appended to {results_path('2d')}")
